In [ ]:
# ========== 导入：笔记本文档化基准测试要用的工具箱 ==========

# 标准库 os：读环境变量（Environment Variables），例如 HF_TOKEN、OPENAI_API_KEY
import os
# 标准库 json：把 .ipynb 当 JSON 解析，提取代码单元格
import json
# 标准库 re：用正则从裁判（judge）回复里抠出 score 数字
import re
# 标准库 time：用 perf_counter 测墙钟延迟（含网络）
import time
# 标准库 traceback：异常时把堆栈格式化进 Gradio 报告，方便排查
import traceback
# Path：用面向对象方式读写文件路径
from pathlib import Path

# load_dotenv：把 .env 里的密钥读进进程环境，避免把密钥写进笔记本
from dotenv import load_dotenv
# gradio：快速搭上传笔记本 + 跑基准测试的 Web UI
import gradio as gr
# ollama：调用本机 Ollama 聊天接口（本地生成方）
import ollama
# OpenAI 客户端：既可打官方 API，也可指向 Hugging Face router
from openai import OpenAI


In [ ]:
# ========== 环境与模型常量：密钥一次加载，模型 id 集中声明 ==========

# 加载 .env：后续 getenv 才能读到密钥（不写进笔记本正文）
load_dotenv()

# 提前读 HF_TOKEN：UI 请求中途才发现缺密钥会像「卡住」，这里 fail-fast
HF_TOKEN = os.getenv("HF_TOKEN", "")
# 把 OPENAI_API_KEY 写回环境：OpenAI() 默认客户端会从这里取密钥
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")

# 裁判模型：要稳、要够强；它决定谁赢（model id 字符串禁止改译）
OPENAI_JUDGE_MODEL = "gpt-4o-mini"

# Nano 作为第三个生成方，和本地 Llama、HF free 同台竞技
OPENAI_NANO_MODEL = "gpt-4.1-nano"

# 本地 Ollama 生成模型；标签写死，避免默默漂到别的 tag
LLAMA_MODEL = "llama3.1:8b"

# HF 免费远程生成模型（经 router）；有配额/限速
HF_FREE_CHAT_MODEL = "HuggingFaceTB/SmolLM3-3B:hf-inference"
# Hugging Face OpenAI 兼容路由基址（URL 禁止改译）
HF_ROUTER_BASE_URL = "https://router.huggingface.co/v1"

# 两个客户端：openai 打裁判+Nano；hf_client 打免费生成方
openai = OpenAI()
hf_client = OpenAI(base_url=HF_ROUTER_BASE_URL, api_key=HF_TOKEN)


In [ ]:
# ========== 从 .ipynb 抽出代码：只喂 code cells，去掉 markdown 噪音 ==========

def notebook_code_extractor(path: str) -> str:
    # 把笔记本当 JSON 读入；encoding 显式 utf-8，避免平台默认编码坑
    nb = json.loads(Path(path).read_text(encoding="utf-8"))

    # 按单元格顺序收集，保证上下文和真实执行顺序一致
    parts = []
    for cell in nb.get("cells", []):
        # 非 code 格跳过：说明文字会干扰「加注释/摘要」任务
        if cell.get("cell_type") != "code":
            continue

        # .ipynb 的 source 常是按行的字符串数组，这里拼成整段
        parts.append("".join(cell.get("source", [])))

    # 单元格之间空一行：函数边界对模型更清晰
    return "\n\n".join(parts).strip()


In [ ]:
# ========== Prompt 组装：指令固定，裁判才在同一规则下比输出 ==========

# 注释任务的 system：要求只回代码（prompt 原文禁止改译）
system_message_comments = (
    "You are a senior developer. Improve the code documentation by adding docstrings and short, useful comments. "
    "Keep it natural and practical. Do not over-comment obvious lines. "
    "Reply with code only."
)

# 摘要任务的 system：只要纯文本摘要，不要代码、不要 Markdown
system_message_summary = (
    "You are a senior developer. Summarize the code clearly: what it does, overall flow, inputs/outputs, and key points. "
    "Do not show the code. Do not use Markdown. Reply with plain text only."
)

def user_prompt_for(code: str) -> str:
    # 统一、明确的 user 指令，降低不同后端之间的方差
    return "Add docstrings and helpful comments. Reply with code only.\n\n" + code

def user_prompt_for_summary(code: str) -> str:
    # 摘要 prompt 单独一条，减少模型把代码「漏」进摘要
    return "Summarize this code.\n\n" + code

def messages_for(code: str):
    # system+user：OpenAI 与 HF router 的 chat completions 都支持
    return [
        {"role": "system", "content": system_message_comments},
        {"role": "user", "content": user_prompt_for(code)},
    ]

def messages_for_summary(code: str):
    # 摘要版 messages：同样是 system + user 两段
    return [
        {"role": "system", "content": system_message_summary},
        {"role": "user", "content": user_prompt_for_summary(code)},
    ]


In [ ]:
# ========== 三个生成后端：Llama 本地 / HF 免费 / GPT Nano ==========

def call_llama_local(code: str):
    # Ollama 跑本地：模型 pull 好之后这条路径「免费」
    r1 = ollama.chat(model=LLAMA_MODEL, messages=messages_for(code))
    r2 = ollama.chat(model=LLAMA_MODEL, messages=messages_for_summary(code))

    # 统一成纯字符串返回，后面流水线不关心后端差异
    return r1["message"]["content"], r2["message"]["content"]

def call_hf_free(code: str):
    # HF router 远程调用：需要 HF_TOKEN，且受配额/延迟影响
    if not HF_TOKEN:
        # 错误文案依赖原样展示，禁止改译
        raise RuntimeError("HF_TOKEN is not set in your environment.")

    # 第一次：生成带注释代码；max_tokens 封顶防大笔记本拖垮延迟/费用
    c1 = hf_client.chat.completions.create(
        model=HF_FREE_CHAT_MODEL,
        messages=messages_for(code),
        max_tokens=1000,  # Cap output so latency/cost don’t explode on large notebooks.
    )
    # 第二次：生成摘要，同样封顶
    c2 = hf_client.chat.completions.create(
        model=HF_FREE_CHAT_MODEL,
        messages=messages_for_summary(code),
        max_tokens=1000,
    )

    # OpenAI 兼容响应：取第一条 choice 的 message.content
    return c1.choices[0].message.content, c2.choices[0].message.content

def call_gpt_nano(code: str):
    # Nano 只当生成方参赛；更强的裁判单独固定，降低自评偏差
    c1 = openai.chat.completions.create(model=OPENAI_NANO_MODEL, messages=messages_for(code))
    c2 = openai.chat.completions.create(model=OPENAI_NANO_MODEL, messages=messages_for_summary(code))
    return c1.choices[0].message.content, c2.choices[0].message.content


In [ ]:
# ========== 单次跑模型：按名字路由 + 测耗时 ==========

def run_model_once(model_name: str, code: str):
    # perf_counter：墙钟时间，远程模型会把网络往返算进去
    t0 = time.perf_counter()

    # 规范化名字：去空白、小写，再用前缀判断走哪条调用链
    m = (model_name or "").strip().lower()
    if m.startswith("llama"):
        # 本地 Llama
        commented, summary = call_llama_local(code)
    elif m.startswith("hf"):
        # Hugging Face free
        commented, summary = call_hf_free(code)
    else:
        # 其余默认走 GPT Nano（例如 "GPT (nano)"）
        commented, summary = call_gpt_nano(code)

    # 第三个返回值是秒数：后面算 score/time（性价比）只用这一个数
    return commented, summary, (time.perf_counter() - t0)


In [ ]:
# ========== LLM 裁判：固定评分格式，便于机器解析 ==========

def _extract_score(text: str) -> float:
    # 解析故意宽松：只要匹配到第一处 score: X / score = X
    m = re.search(r"\bscore\s*[:=]\s*([0-9]+(?:\.[0-9]+)?)", text, flags=re.IGNORECASE)
    # 匹配失败就当 0 分，避免整条流水线崩掉
    return float(m.group(1)) if m else 0.0

def judge_quality_llm(code: str, commented: str, summary: str) -> str:
    # 把裁判输出格式锁死，基准测试才能稳定 parse（rubric 原文禁止改译）
    rubric = (
        "You are a strict code reviewer. Evaluate the assistant output for the given original code.\n"
        "Return a short verdict with a single numeric score from 0 to 10.\n"
        "Criteria (equal weight):\n"
        "1) Correctness: comments/docstrings match what code does (no hallucinations).\n"
        "2) Usefulness: captures intent, assumptions, edge cases, and non-obvious behavior.\n"
        "3) Clarity: readable, consistent, avoids redundant commentary.\n"
        "4) Naturalness: reads like a human developer wrote it.\n"
        "Output format (exact):\n"
        "score: <number>\n"
        "notes: <one paragraph>\n"
    )

    # 最小上下文：原文 + 注释版 + 摘要，三者齐才能打质量分
    payload = (
        "ORIGINAL CODE:\n"
        f"{code}\n\n"
        "COMMENTED CODE:\n"
        f"{commented}\n\n"
        "SUMMARY:\n"
        f"{summary}\n"
    )

    # 裁判也是 chat：system=评分细则，user=待评材料
    messages = [
        {"role": "system", "content": rubric},
        {"role": "user", "content": payload},
    ]

    # 裁判固定 gpt-4o-mini：靶子不动，只让参赛生成方变化
    c = openai.chat.completions.create(model=OPENAI_JUDGE_MODEL, messages=messages, max_tokens=400)
    return c.choices[0].message.content


In [ ]:
# ========== 基准主流程：三方生成 → 裁判打分 → 按性价比选赢家 ==========

def benchmark_and_pick_winner(file_obj):
    # Gradio 回调：返回 (报告, 抽出的代码, 赢家注释代码, 赢家摘要)
    try:
        # 没上传文件：直接返回错误文案（英文原样，UI 依赖）
        if file_obj is None:
            return "ERROR: Please upload a .ipynb file.", "", "", ""

        # Gradio 上传后会在磁盘给临时路径；.name 即该路径
        path = file_obj.name
        # 扩展名校验：只接受 .ipynb
        if not path.lower().endswith(".ipynb"):
            return "ERROR: The uploaded file is not a .ipynb notebook.", "", "", ""

        # 代码只抽一次：三个参赛方看到完全相同的输入
        code = notebook_code_extractor(path)
        if not code.strip():
            return "ERROR: No code cells found in the notebook.", "", "", ""

        # 三个参赛方：免费远程 / 本地 / 便宜云端
        candidates = ["HF (free)", "Llama (local)", "GPT (nano)"]
        # name → 结果字典
        results = {}

        for name in candidates:
            # 生成注释版+摘要，并计量时
            commented, summary, secs = run_model_once(name, code)

            # 独立裁判打质量分，才能跨生成方比较
            verdict = judge_quality_llm(code, commented, summary)
            score = _extract_score(verdict)

            # score/time：又好又快的模型更吃香
            value = (score / secs) if secs > 0 else 0.0

            # 记下该参赛方的全部指标，后面写报告与挑赢家
            results[name] = {
                "commented": commented,
                "summary": summary,
                "secs": secs,
                "verdict": verdict,
                "score": score,
                "value": value,
            }

        # 先比 value，再比 raw score：耗时接近时质量优先
        winner = max(results.items(), key=lambda kv: (kv[1]["value"], kv[1]["score"]))[0]
        w = results[winner]

        # 拼一份紧凑可读的报告，说明为什么选这个赢家
        lines = []
        for name in candidates:
            r = results[name]
            lines.append(f"{name}: score={r['score']:.2f}, time={r['secs']:.3f}s, score/time={r['value']:.3f}")
        lines.append("")
        lines.append(f"WINNER: {winner}")
        lines.append("")
        lines.append("Judge verdict (winner):")
        lines.append(w["verdict"].strip())

        # 四元组回填 Gradio 四个输出控件
        return "\n".join(lines), code, w["commented"], w["summary"]

    except Exception:
        # 把 traceback 送进 UI：可调试，又不会把 Gradio 队列打崩
        return "ERROR:\n" + traceback.format_exc(), "", "", ""


In [ ]:
# ========== Gradio UI：上传笔记本 → 生成并评选赢家 ==========

# 自定义 CSS：给「注释代码 / 摘要」两块不同底色（字符串原样保留）
css = """
.comments {background-color: #00599C;}
.summary {background-color: #008B8B;}
"""

# Blocks：用 with 上下文搭页面；css 注入上面样式
with gr.Blocks(css=css) as ui:
    # 顶部说明（UI 文案是可运行字符串，保持英文原样）
    gr.Markdown(
        "### Notebook Documentation Tool\n"
        "Upload a notebook, generate docs with three models, and rank them with a GPT-4o-mini judge."
    )

    # 文件上传：限制 .ipynb
    with gr.Row():
        nb_file = gr.File(label="Upload .ipynb", file_types=[".ipynb"])

    # 主按钮：触发 benchmark_and_pick_winner
    with gr.Row():
        run_bench = gr.Button("Generate and pick winner")

    # 基准报告文本框
    with gr.Row():
        report = gr.Textbox(label="Benchmark report", lines=10)

    # 只读展示抽出的笔记本代码
    with gr.Row():
        source_code = gr.Textbox(label="Extracted notebook code (read-only)", lines=14, interactive=False)

    # 并排：赢家的注释代码 + 摘要（用 elem_classes 套 CSS）
    with gr.Row():
        commented_code = gr.Textbox(label="Winner: documented code", lines=14, elem_classes=["comments"])
        code_summary = gr.Textbox(label="Winner: summary", lines=14, elem_classes=["summary"])

    # 点击：输入文件 → 四个输出控件
    run_bench.click(
        benchmark_and_pick_winner,
        inputs=[nb_file],
        outputs=[report, source_code, commented_code, code_summary],
    )

# 启动 Gradio；inbrowser=True 尝试自动打开浏览器
ui.launch(inbrowser=True)
